# Qwen2.5-7B full fine-tune — RAG reliability judge (Method 1 / 2)
Full fine-tuning (not LoRA) of `Qwen/Qwen2.5-7B-Instruct`. Auto-detects GPU and
picks a memory strategy. Set `MODE` in the config cell to `"direct"` or `"marker"`.
Runs on Colab, Kaggle, Yandex DataSphere.

In [ ]:
# --- Install pinned deps (skip if already present) ---
import subprocess, sys

PKGS = [
    "transformers>=4.56.2",
    "trl==1.8.0",
    "accelerate>=1.4.0",
    "datasets>=4.7.0",
    "peft>=0.8.0",
    "bitsandbytes>=0.44.1",
    "deepspeed>=0.14.4",
    "sentencepiece",
    "psutil",
]
def _pip(args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
try:
    import trl, transformers  # noqa: F401
except Exception:
    _pip(PKGS)

# --- Detect platform ---
import os
if "google.colab" in sys.modules or os.path.exists("/content"):
    PLATFORM = "colab"
elif os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
elif os.path.exists("/home/jupyter") or "DATASPHERE" in os.environ.get("HOSTNAME", "").upper():
    PLATFORM = "datasphere"
else:
    PLATFORM = "other"
print("platform:", PLATFORM)

In [ ]:
# --- Get the repo so training == inference format ---
REPO_URL = "https://github.com/aldem2k00/rag-reliability.git"  # set to your fork/remote
REPO_BRANCH = "qwen7b-notebook"  # branch that contains rag_reliability.nb_format; "" = default branch
REPO_DIR = "rag-reliability"
import os, subprocess, sys

def _have_repo():
    try:
        import rag_reliability  # noqa: F401
        return True
    except Exception:
        return False

if not _have_repo():
    if not os.path.exists(REPO_DIR):
        clone_cmd = ["git", "clone", "-q", "--depth", "1"]
        if REPO_BRANCH:
            clone_cmd += ["-b", REPO_BRANCH]
        clone_cmd += [REPO_URL, REPO_DIR]
        try:
            subprocess.check_call(clone_cmd)
        except Exception as e:
            print("clone failed, using inline fallback:", e)
    # src-layout: add the package dir to sys.path directly (robust vs. editable-install
    # quirks on Colab/Kaggle). Core deps (pydantic, sklearn, numpy, pyyaml) are preinstalled.
    src_path = os.path.abspath(os.path.join(REPO_DIR, "src"))
    if os.path.isdir(src_path) and src_path not in sys.path:
        sys.path.insert(0, src_path)

try:
    from rag_reliability.nb_format import build_sft_messages
    from rag_reliability.dataset import load_jsonl, split_samples
    from rag_reliability.parsing import parse_prediction
    from rag_reliability.schema import RagSample
    from rag_reliability import metrics as M
    USING_REPO = True
except Exception as e:
    print("repo import failed -> inline fallback:", e)
    USING_REPO = False
    # Paste verbatim: nb_format.INLINE fallback (kept in sync by tests/test_nb_format.py)
    # NOTE for implementer: copy the bodies of build_direct_prompt/build_marker_prompt,
    # build_direct_target/build_marker_target, resolve_marker, RagSample, ALLOWED_MARKERS,
    # parse_prediction, load_jsonl, split_samples here. Source of truth: the repo modules.
    raise RuntimeError("Set REPO_URL to a reachable remote, or paste the inline fallback block.")

print("format source:", "repo" if USING_REPO else "inline")

In [ ]:
# --- Config: edit these ---
MODE = "direct"              # "direct" (Method 1) or "marker" (Method 2)
assert MODE in ("direct", "marker")
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

USE_REAL_DATA = False        # False -> dummy smoke set; True -> your organizers.jsonl
DATA_PATH = (f"{REPO_DIR}/data/dummy.jsonl" if not USE_REAL_DATA
             else "data/organizers.jsonl")

EPOCHS = 3
LR = 1e-5                    # full FT wants a smaller LR than LoRA
MAX_SEQ_LEN = 2048
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
SEED = 42

SAVE_TARGET = "auto"        # "auto" -> Drive on Colab / working dir elsewhere; or a path
QLORA_FALLBACK = False      # True only if hardware can't do full FT (NOT full FT)
OUTPUT_DIR = f"ft_{MODE}"
print(dict(MODE=MODE, model=BASE_MODEL, data=DATA_PATH, qlora=QLORA_FALLBACK))


In [ ]:
# --- Detect GPU/RAM, pick a real full-FT profile ---
import torch, psutil

N_GPUS = torch.cuda.device_count()
per_gpu = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(N_GPUS)]
TOTAL_VRAM_GB = sum(per_gpu)
MIN_GPU_GB = min(per_gpu) if per_gpu else 0.0
CPU_RAM_GB = psutil.virtual_memory().total / 1e9

MULTI_PROC = False
if TOTAL_VRAM_GB >= 70 and N_GPUS == 1:
    PROFILE = "full_single"
elif N_GPUS == 1 and MIN_GPU_GB >= 22:           # 1x40GB single-proc ZeRO-3 offload
    PROFILE = "full_zero3_offload"
elif N_GPUS >= 2:                                # 2xT4 / 2x24GB / 2x40GB: shard via notebook_launcher
    PROFILE, MULTI_PROC = "full_zero3_offload", True
else:
    PROFILE = "insufficient"

if PROFILE == "full_zero3_offload" and CPU_RAM_GB < 55:
    print(f"WARNING: ZeRO-3 CPU offload wants >=~60GB RAM, have {CPU_RAM_GB:.0f}GB — may OOM.")

if PROFILE == "insufficient" and not QLORA_FALLBACK:
    raise RuntimeError(
        f"Full FT of 7B needs more than {TOTAL_VRAM_GB:.0f}GB VRAM across {N_GPUS} GPU(s). "
        "Use an 80GB A100/H100, a 40GB card, or 2xT4 — or set QLORA_FALLBACK=True (NOT full FT)."
    )
print(dict(profile=PROFILE, gpus=N_GPUS, vram_gb=round(TOTAL_VRAM_GB), ram_gb=round(CPU_RAM_GB),
           multi_proc=MULTI_PROC, qlora=QLORA_FALLBACK))


In [ ]:
# --- Load, split (stratified by reliable, seed 42), build chat records ---
from datasets import Dataset

samples = load_jsonl(DATA_PATH)
train, val, test_samples = split_samples(samples, seed=SEED)
print(f"loaded {len(samples)} -> train={len(train)} val={len(val)} test={len(test_samples)}")

def _to_ds(rows):
    return Dataset.from_list([build_sft_messages(s, MODE) for s in rows])

train_ds, val_ds = _to_ds(train), _to_ds(val)

# In-notebook symmetry assertion (same guard as tests/test_nb_format.py)
from rag_reliability.formatting import build_chat_training_record
assert train_ds[0] == build_chat_training_record(train[0], MODE), "format drift!"
print("format symmetry OK. sample:", train_ds[0]["messages"][1]["content"])

In [ ]:
# --- Model + tokenizer loader (called inside the train fn) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tokenizer():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(torch_dtype=torch.bfloat16)
    if QLORA_FALLBACK:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
    # ZeRO-3 sets device placement itself; full_single and QLoRA need explicit placement.
    if QLORA_FALLBACK or PROFILE == "full_single":
        kwargs["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    model.config.use_cache = False
    return model, tok

print("loader ready for profile:", PROFILE)

In [ ]:
# --- DeepSpeed ZeRO-3 offload config (used by both offload paths) ---
import json
ZERO3 = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True, "contiguous_gradients": True,
        "stage3_gather_16bit_weights_on_model_save": True,
    },
    "gradient_accumulation_steps": GRAD_ACCUM,
    "train_micro_batch_size_per_gpu": PER_DEVICE_BATCH,
    "gradient_clipping": 1.0,
}
with open("ds_zero3.json", "w") as f: json.dump(ZERO3, f, indent=2)

def train_fn():
    from trl import SFTConfig, SFTTrainer
    model, tok = load_model_and_tokenizer()
    if QLORA_FALLBACK:
        from peft import LoraConfig, prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)
        peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                              target_modules=["q_proj","k_proj","v_proj","o_proj",
                                              "gate_proj","up_proj","down_proj"])
    else:
        peft_cfg = None

    cfg = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        optim=("adamw_torch" if (PROFILE == "full_zero3_offload" and not QLORA_FALLBACK) else "adamw_bnb_8bit"),
        max_length=MAX_SEQ_LEN,        # trl>=1.x renamed max_seq_length -> max_length
        bf16=True,
        gradient_checkpointing=True,
        assistant_only_loss=True,      # == mlx --mask-prompt: loss on assistant tokens only
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch",
        per_device_eval_batch_size=1,
        report_to="none",
        seed=SEED,
        deepspeed=("ds_zero3.json" if PROFILE == "full_zero3_offload" and not QLORA_FALLBACK else None),
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
                         eval_dataset=val_ds, processing_class=tok, peft_config=peft_cfg)
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tok.save_pretrained(OUTPUT_DIR)

# --- Launch ---
if MULTI_PROC:
    from accelerate import notebook_launcher
    notebook_launcher(train_fn, num_processes=N_GPUS)
else:
    train_fn()
print("training done ->", OUTPUT_DIR)


In [ ]:
# --- Evaluate: greedy generate on test, parse, score with repo metrics ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rag_reliability.prompts import build_direct_prompt, build_marker_prompt

import gc
gc.collect()
torch.cuda.empty_cache()

build_prompt = build_direct_prompt if MODE == "direct" else build_marker_prompt
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16,
                                             device_map="auto")
model.eval()

@torch.no_grad()
def generate(prompt: str) -> str:
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=64, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

preds = []
for s in test_samples:
    raw = generate(build_prompt(s))
    preds.append(parse_prediction(raw, s.id, expect_marker=(MODE == "marker")))

# metrics.evaluate_predictions(samples, predictions) -> EvaluationResult
# (confirmed signature; it already computes invalid_output_rate internally).
metrics_out = M.evaluate_predictions(test_samples, preds)
print(metrics_out.model_dump_json(indent=2) if hasattr(metrics_out, "model_dump_json") else metrics_out)

In [ ]:
# --- Save checkpoint + print how to run it through the repo pipeline ---
import shutil, os

def resolve_dest():
    if SAVE_TARGET != "auto":
        return SAVE_TARGET
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
        return f"/content/drive/MyDrive/{OUTPUT_DIR}"
    if PLATFORM == "kaggle":
        return f"/kaggle/working/{OUTPUT_DIR}"
    return os.path.abspath(OUTPUT_DIR)

dest = resolve_dest()
if os.path.abspath(dest) != os.path.abspath(OUTPUT_DIR):
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
print("saved to:", dest)

# Optional HF Hub push (off by default):
# from huggingface_hub import login; login(token="hf_..."); model.push_to_hub("you/qwen7b-judge-"+MODE)

print(f"""
Next — run this checkpoint through the repo pipeline:
  python scripts/infer.py --data data/dummy.jsonl --mode {MODE} \\
    --model {dest} --output results/{MODE}_ft_predictions.jsonl
  python scripts/evaluate.py --data data/dummy.jsonl \\
    --predictions results/{MODE}_ft_predictions.jsonl --output results/{MODE}_ft_metrics.json
""")
